In [1]:
from typing import Dict
import hydra
import torch
import pathlib
import train
import numpy as np
from contextlib import contextmanager
import tqdm

@contextmanager
def suppress_prints():
    """Context manager to suppress all print statements."""
    original_print = __builtins__.print
    __builtins__.print = lambda *args, **kwargs: None
    try:
        yield original_print
    finally:
        __builtins__.print = original_print

checkpoint_directories = {
    "mlp_symmetry0": "outputs/2025-12-17/12-18-16_mlp_mnist_sym-0__3pb1rw8b__kk13eg0q",
    "mlp_symmetry1_kappa0": "outputs/2025-12-17/14-20-51_mlp_mnist_sym-1__m3z8jvf9__iwzo2u22",
    "mlp_symmetry1_kappa1": "outputs/2025-12-17/12-30-55_mlp_mnist_sym-1__3pb1rw8b__4b0gt1xm",
    "mlp_symmetry1_kappaPerLayer": "outputs/2025-12-17/12-46-23_mlp_mnist_sym-1__3pb1rw8b__8fvvk8f3",
    "mlp_symmetry2": "outputs/2025-12-17/13-01-05_mlp_mnist_sym-2__3pb1rw8b__48q9fddu",
    "mlp_symmetry3_kappa0": "outputs/2025-12-17/13-16-26_mlp_mnist_sym-3__3pb1rw8b__tq3utpfq",
    "mlp_symmetry3_kappa1": "outputs/2025-12-17/13-30-38_mlp_mnist_sym-3__3pb1rw8b__ruzzxkpy",
    "mlp_symmetry3_kappaPerLayer": "outputs/2025-12-17/13-45-10_mlp_mnist_sym-3__3pb1rw8b__e4vv3n8v",
}

# TODO generalize this: currently takes all params that don't have "mask" in their name (mask params in WLMPs) and that are not of the form "activations.<...>.C" (fixed params in sigma-MLPs). Ok?
def normalized_directional_curvature(model, loss_fn, model_inputs, direction_delta, model_params=None):
    def eval_model(eval_params):
        k = 0
        new_state_dict = {
            name: eval_params[k:(k := k+param.numel())].reshape(param.shape)
            for name, param in model.state_dict().items()
            if not "mask" in name and not (name.startswith("activations") and name.endswith(".C"))
        }
        out = torch.func.functional_call(model, new_state_dict, model_inputs)
        return loss_fn(out)
    eval_params = model_params if model_params is not None else torch.cat([p.view(-1) for p in model.state_dict().values()])
    _, hessian_delta_dot_product = torch.autograd.functional.vhp(eval_model, eval_params, direction_delta)
    return (hessian_delta_dot_product * direction_delta).sum() / torch.linalg.norm(direction_delta)**2

def compute_linear_path_normalized_curvatures(checkpoint_path_1, checkpoint_path_2, steps: int, device="cuda:0", data_info=None):
    with hydra.initialize(version_base=None, config_path=str(pathlib.Path(checkpoint_path_1).parent)):
        cfg = hydra.compose(config_name="config")
    cfg.dataset.batch_size = 10000
    if data_info is None:
        data_info = train.setup_data_loaders(cfg)

    model1, model2 = [train.create_model(cfg, mask_seed=-1).to(device) for _ in range(2)]
    model1.load_state_dict(torch.load(checkpoint_path_1, map_location=device)["model_state_dict"])
    model2.load_state_dict(torch.load(checkpoint_path_2, map_location=device)["model_state_dict"])

    interpolation_factors = torch.linspace(0, 1, steps=steps).tolist()
    results: Dict[str, Dict[float, float]] = {
        split: {alpha: 0 for alpha in interpolation_factors} for split in ["train", "val", "test"]}

    model1_params, model2_params = [torch.cat([p.view(-1) for p in model.parameters()]) for model in [model1, model2]]

    print(model1_params.shape)
    print(sum(p.numel() for _, p in model1.state_dict().items()))
    print(sum(p.numel() for n, p in model1.state_dict().items() if "mask" not in n))
    print(model1.state_dict().keys())

    for split_name, loader in [('train', data_info["train_loader"]), ('val', data_info["val_loader"]), ('test', data_info["test_loader"])]:
        if loader is None:
            continue

        total_instances = 0

        with torch.no_grad():
            for batch in tqdm.tqdm(loader, desc=split_name):
                if isinstance(batch, (list, tuple)):
                    data, target = batch
                else:
                    data = batch.x
                    target = batch.y.squeeze()

                data, target = data.to(device), target.to(device)
                total_instances += len(data)

                loss_fn = lambda out: torch.nn.functional.cross_entropy(out, target)
                # TODO check if this kind of averaging is valid!
                for alpha in interpolation_factors:
                    curvature_sum = len(batch) * normalized_directional_curvature(
                        model1, loss_fn, data,
                        direction_delta=model2_params-model1_params,
                        model_params=model2_params*alpha+model1_params*(1-alpha)
                    ).item()
                    results[split_name][alpha] += curvature_sum
        results[split_name] = {alpha: curvature_sum / total_instances for alpha, curvature_sum in results[split_name].items()}

    return results

results = {
    run_key: 
        compute_linear_path_normalized_curvatures(
        f"{checkpoint_directories[run_key]}/checkpoint_epoch_100_model_1.pt",
        f"{checkpoint_directories[run_key]}/checkpoint_epoch_100_model_2.pt",
        steps=301,
    )
    for run_key in ["mlp_symmetry1_kappa0", "mlp_symmetry1_kappa1", "mlp_symmetry2"]
}

Mask params for linear0: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear1: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear2: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear3: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 256}
Mask params for linear0: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear1: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear2: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear3: {'mask_constant': 0.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 256}
torch.Size([935434])
2797066
935434
od

test: 100%|██████████| 1/1 [00:04<00:00,  4.50s/it]


Mask params for linear0: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear1: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear2: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear3: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 256}
Mask params for linear0: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear1: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear2: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 64}
Mask params for linear3: {'mask_constant': 1.0, 'mask_type': 'random_subsets', 'do_normal_mask': True, 'num_fixed': 256}
torch.Size([935434])
2797066
935434
od

test: 100%|██████████| 1/1 [00:04<00:00,  4.56s/it]


torch.Size([935434])
1721866
1721866
odict_keys(['lins.0.weight', 'lins.0.bias', 'lins.1.weight', 'lins.1.bias', 'lins.2.weight', 'lins.2.bias', 'lins.3.weight', 'lins.3.bias', 'activations.0.C', 'activations.1.C', 'activations.2.C', 'norms.0.weight', 'norms.0.bias', 'norms.1.weight', 'norms.1.bias', 'norms.2.weight', 'norms.2.bias'])


test: 100%|██████████| 1/1 [00:05<00:00,  5.22s/it]


In [26]:
import pandas as pd
import plotly.express as px

RUN_DESCRIPTIONS = {
    "mlp_symmetry1_kappa1": "WMLP with large fixed weights",
    "mlp_symmetry1_kappa0": "WMLP with zero fixed weights",
    "mlp_symmetry2": "Sigma MLP"
}

result_dicts = [
    dict(split=split, alpha=alpha, curvature=curvature, run_key=run, run_desc=RUN_DESCRIPTIONS[run])
    for run, run_results in results.items()
    for split, split_results in run_results.items()
    for alpha, curvature in split_results.items()
]
df = pd.DataFrame(result_dicts)
px.line(df[df["split"] == "train"], x="alpha", y="curvature", color="run_desc", title="Normalized curvature along LMC path, WMLP with large fixed weights", height=700)